In [1]:
# CELL 1
# Install the MySQL connector for Python and create the ravenswood_operations database.
# We use sqlalchemy as the connection engine — it integrates cleanly with pandas
# and allows us to load dataframes directly into MySQL tables.
# The password is entered via a secure prompt so it is never stored in the notebook.

import subprocess
import sys

# Install required packages if not already installed
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sqlalchemy',
                'pymysql', 'cryptography'], capture_output=True)

import pymysql
from sqlalchemy import create_engine, text
import pandas as pd
import getpass
import os

# Connection details
HOST = 'localhost'
PORT = 3306
USER = 'root'
DB_NAME = 'ravenswood_operations'

# Secure password prompt — never hardcode passwords in notebooks
PASSWORD = getpass.getpass('Enter your MySQL root password: ')

# Connect without specifying database first to create it
engine_root = create_engine(
    f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/',
    echo=False
)

# Create database if it doesn't exist
with engine_root.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    conn.execute(text(f"USE {DB_NAME}"))
    print(f"Database '{DB_NAME}' created successfully")

# Now connect directly to the database
engine = create_engine(
    f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}',
    echo=False
)

print(f"Connected to {HOST}:{PORT}/{DB_NAME}")
print("Ready to load tables")

Database 'ravenswood_operations' created successfully
Connected to localhost:3306/ravenswood_operations
Ready to load tables


In [2]:
# CELL 2
# Create all tables in the ravenswood_operations database.
# We define the full schema across three pillars plus a reference table.
# Tables are dropped and recreated if they already exist — this makes the
# notebook safely re-runnable during development.
# Foreign key relationships link all pillar tables through financial_year.

schema_sql = """
-- ============================================================
-- REFERENCE TABLE
-- ============================================================
DROP TABLE IF EXISTS ref_financial_years;
CREATE TABLE ref_financial_years (
    financial_year      VARCHAR(7)      PRIMARY KEY,
    year_start          DATE            NOT NULL,
    year_end            DATE            NOT NULL,
    expansion_period    TINYINT(1)      DEFAULT 0
        COMMENT '1 = Ravenswood expansion period 2020-21 to 2023-24'
);

-- ============================================================
-- PILLAR 1: PRODUCTION
-- ============================================================
DROP TABLE IF EXISTS production_summary;
CREATE TABLE production_summary (
    id                      INT             AUTO_INCREMENT PRIMARY KEY,
    financial_year          VARCHAR(7)      NOT NULL,
    ore_throughput_tonnes   BIGINT,
    dore_output_tonnes      DECIMAL(10,3),
    gold_mined_oz           DECIMAL(12,2),
    gold_recovered_oz       DECIMAL(12,2),
    ore_to_dore_ratio       DECIMAL(12,2),
    recovery_rate_pct       DECIMAL(6,2),
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

-- ============================================================
-- PILLAR 2: LABOUR
-- ============================================================
DROP TABLE IF EXISTS labour_quarterly;
CREATE TABLE labour_quarterly (
    id                              INT             AUTO_INCREMENT PRIMARY KEY,
    quarter_date                    DATE            NOT NULL,
    quarter_label                   VARCHAR(50),
    financial_year                  VARCHAR(7)      NOT NULL,
    total_surface_mineral_workers   INT,
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

DROP TABLE IF EXISTS labour_production_linked;
CREATE TABLE labour_production_linked (
    id                      INT             AUTO_INCREMENT PRIMARY KEY,
    financial_year          VARCHAR(7)      NOT NULL,
    ore_throughput_tonnes   BIGINT,
    dore_output_tonnes      DECIMAL(10,3),
    gold_mined_oz           DECIMAL(12,2),
    gold_recovered_oz       DECIMAL(12,2),
    ore_to_dore_ratio       DECIMAL(12,2),
    recovery_rate_pct       DECIMAL(6,2),
    avg_workers             INT,
    max_workers             INT,
    min_workers             INT,
    tonnes_per_worker       DECIMAL(12,2),
    workers_per_mtpa        DECIMAL(8,2),
    gold_oz_per_worker      DECIMAL(10,2),
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

-- ============================================================
-- PILLAR 3: SAFETY
-- ============================================================
DROP TABLE IF EXISTS safety_monthly;
CREATE TABLE safety_monthly (
    id                      INT             AUTO_INCREMENT PRIMARY KEY,
    month                   DATE            NOT NULL,
    financial_year          VARCHAR(7)      NOT NULL,
    serious_accidents       INT,
    hpi_count               INT,
    million_hours_worked    DECIMAL(8,4),
    safr                    DECIMAL(8,4),
    safr_monthly_avg        DECIMAL(8,4),
    safr_3mth_rolling       DECIMAL(8,4),
    safr_12mth_rolling      DECIMAL(8,4),
    hpifr                   DECIMAL(8,4),
    hpifr_monthly_avg       DECIMAL(8,4),
    hpifr_3mth_rolling      DECIMAL(8,4),
    hpifr_12mth_rolling     DECIMAL(8,4),
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

DROP TABLE IF EXISTS safety_annual;
CREATE TABLE safety_annual (
    id                          INT             AUTO_INCREMENT PRIMARY KEY,
    financial_year              VARCHAR(7)      NOT NULL,
    total_serious_accidents     INT,
    total_hpi_count             INT,
    total_million_hours         DECIMAL(8,2),
    avg_safr                    DECIMAL(8,4),
    avg_hpifr                   DECIMAL(8,4),
    peak_safr                   DECIMAL(8,4),
    peak_hpifr                  DECIMAL(8,4),
    months_reported             INT,
    safr_12mth_annual           DECIMAL(8,4),
    hpifr_12mth_annual          DECIMAL(8,4),
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

DROP TABLE IF EXISTS safety_production_linked;
CREATE TABLE safety_production_linked (
    id                          INT             AUTO_INCREMENT PRIMARY KEY,
    financial_year              VARCHAR(7)      NOT NULL,
    total_serious_accidents     INT,
    total_hpi_count             INT,
    avg_safr                    DECIMAL(8,4),
    avg_hpifr                   DECIMAL(8,4),
    ore_throughput_tonnes       BIGINT,
    recovery_rate_pct           DECIMAL(6,2),
    sa_per_million_tonnes       DECIMAL(8,4),
    hpi_per_million_tonnes      DECIMAL(8,4),
    hpi_to_sa_ratio             DECIMAL(8,2),
    FOREIGN KEY (financial_year) REFERENCES ref_financial_years(financial_year)
);

-- ============================================================
-- REFERENCE TABLE: SKILL SHORTAGES
-- ============================================================
DROP TABLE IF EXISTS ref_skill_shortages;
CREATE TABLE ref_skill_shortages (
    id                  INT             AUTO_INCREMENT PRIMARY KEY,
    process_step        VARCHAR(50)     NOT NULL
        COMMENT 'Mining process step: Dig, Crush, Leach, Smelt, Maintenance, Technical',
    occupation          VARCHAR(100)    NOT NULL,
    anzsco_code         VARCHAR(10),
    jsa_shortage_2021   VARCHAR(20),
    jsa_shortage_2022   VARCHAR(20),
    jsa_shortage_2023   VARCHAR(20),
    jsa_shortage_2024   VARCHAR(20),
    jsa_shortage_2025   VARCHAR(20),
    emp_per_mtpa_low    DECIMAL(5,1)
        COMMENT 'Lower bound employees per million tonnes per annum',
    emp_per_mtpa_high   DECIMAL(5,1)
        COMMENT 'Upper bound employees per million tonnes per annum',
    notes               VARCHAR(255)
);
"""

# Execute schema — split by semicolon and run each statement
with engine.connect() as conn:
    for statement in schema_sql.split(';'):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))
    conn.commit()

print("Schema created successfully. Tables:")
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    for row in result:
        print(f"  ✅ {row[0]}")

Schema created successfully. Tables:
  ✅ labour_production_linked
  ✅ labour_quarterly
  ✅ production_summary
  ✅ ref_financial_years
  ✅ ref_skill_shortages
  ✅ safety_annual
  ✅ safety_monthly
  ✅ safety_production_linked


In [3]:
# CELL 3
# Populate the two reference tables first before loading pillar data.
# ref_financial_years serves as the foreign key anchor for all pillar tables —
# every financial year that appears in any pillar table must exist here first.
# ref_skill_shortages stores the occupation shortage and workforce intensity
# benchmarks we discussed earlier — linking process steps to JSA shortage data.

import pandas as pd
from sqlalchemy import text

# ── ref_financial_years ──────────────────────────────────────────────────
financial_years = pd.DataFrame([
    ('2012-13', '2012-07-01', '2013-06-30', 0),
    ('2013-14', '2013-07-01', '2014-06-30', 0),
    ('2014-15', '2014-07-01', '2015-06-30', 0),
    ('2015-16', '2015-07-01', '2016-06-30', 0),
    ('2016-17', '2016-07-01', '2017-06-30', 0),
    ('2017-18', '2017-07-01', '2018-06-30', 0),
    ('2018-19', '2018-07-01', '2019-06-30', 0),
    ('2019-20', '2019-07-01', '2020-06-30', 0),
    ('2020-21', '2020-07-01', '2021-06-30', 1),
    ('2021-22', '2021-07-01', '2022-06-30', 1),
    ('2022-23', '2022-07-01', '2023-06-30', 1),
    ('2023-24', '2023-07-01', '2024-06-30', 1),
    ('2024-25', '2024-07-01', '2025-06-30', 0),
    ('2025-26', '2025-07-01', '2026-06-30', 0),
], columns=['financial_year', 'year_start', 'year_end', 'expansion_period'])

financial_years.to_sql('ref_financial_years', engine,
                       if_exists='append', index=False)
print(f"ref_financial_years loaded: {len(financial_years)} rows")

# ── ref_skill_shortages ──────────────────────────────────────────────────
skill_shortages = pd.DataFrame([
    ('Dig',         'Drillers, Miners & Shot Firers',       '711211',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 4.0,  6.0,
     'Core open cut mining crew — consistently hardest to fill'),
    ('Dig',         'Mobile Plant Operators',               '721111',
     'No Shortage','Shortage','Shortage','Shortage','Shortage', 8.0, 12.0,
     'Haul truck and excavator operators — DIDO premium required'),
    ('Dig',         'Mining Engineers',                     '233611',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 2.0,  3.0,
     'Fill rate 42% in 2021 — 6th lowest of all occupations'),
    ('Dig',         'Geologists',                           '234412',
     'No Shortage','Shortage','Shortage','Shortage','Shortage', 1.0,  2.0,
     'Grade control geologists critical for open cut operations'),
    ('Crush',       'Mechanical Fitters',                   '323211',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 2.0,  3.0,
     'Fixed plant maintenance for crusher and ball mill circuits'),
    ('Crush',       'Electrical & Instrumentation Technicians', '342111',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 1.5,  2.5,
     'Critical for crusher control systems and instrumentation'),
    ('Leach',       'Processing Plant Operators',           '711911',
     'No Shortage','No Shortage','Shortage','Shortage','Shortage', 2.0, 3.0,
     'CIP leach circuit operators — shortage emerged post-expansion'),
    ('Leach',       'Metallurgists',                        '233911',
     'No Shortage','No Shortage','Shortage','Shortage','Shortage', 1.0, 2.0,
     'Process optimisation for gold recovery rate improvement'),
    ('Smelt',       'Laboratory Technicians',               '311411',
     'No Shortage','Shortage','Shortage','Shortage','Shortage', 0.5, 1.0,
     'Assay and quality control for dore production'),
    ('Maintenance', 'Diesel Mechanics',                     '321212',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 3.0,  5.0,
     'Heavy mobile equipment — highest demand role at remote sites'),
    ('Maintenance', 'Electricians',                         '341111',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 2.0,  3.0,
     'Fixed and mobile plant electrical maintenance'),
    ('Technical',   'Surveyors & Spatial Scientists',       '232212',
     'Shortage','Shortage','Shortage','Shortage','Shortage', 0.5, 1.0,
     'Open pit survey and grade control support'),
    ('Technical',   'Data & Business Intelligence Analysts','224111',
     'No Shortage','No Shortage','No Shortage','Shortage','Shortage', 0.5, 1.0,
     'Digitalisation driver — target role for this portfolio'),
    ('Technical',   'Safety & Health Officers',             '251311',
     'No Shortage','Shortage','Shortage','Shortage','Shortage', 1.0, 1.5,
     'Regulatory compliance and incident management'),
    ('Administration','Environmental Scientists',           '234312',
     'No Shortage','No Shortage','No Shortage','Shortage','Shortage', 0.5, 1.0,
     'EA compliance and rehabilitation obligations'),
], columns=[
    'process_step', 'occupation', 'anzsco_code',
    'jsa_shortage_2021', 'jsa_shortage_2022', 'jsa_shortage_2023',
    'jsa_shortage_2024', 'jsa_shortage_2025',
    'emp_per_mtpa_low', 'emp_per_mtpa_high', 'notes'
])

skill_shortages.to_sql('ref_skill_shortages', engine,
                       if_exists='append', index=False)
print(f"ref_skill_shortages loaded: {len(skill_shortages)} rows")

print("\nReference tables ready ✅")

ref_financial_years loaded: 14 rows
ref_skill_shortages loaded: 15 rows

Reference tables ready ✅


In [5]:
# CELL 4
# Load the production pillar CSV into the production_summary table.
# This is the master production table built in Notebook 01 containing
# ore throughput, dore output, gold mined, gold recovered and recovery rate.
# We validate the row count after loading to confirm data integrity.

import pandas as pd
import os

# Load processed CSV
production_path = os.path.join('..', 'data', 'processed', 'production',
                                'gold_production_qld.csv')
df_production = pd.read_csv(production_path)

# Ensure correct dtypes before loading
df_production['ore_throughput_tonnes'] = df_production[
    'ore_throughput_tonnes'].astype('Int64')
df_production['gold_mined_oz'] = df_production['gold_mined_oz'].round(2)
df_production['gold_recovered_oz'] = df_production['gold_recovered_oz'].round(2)
df_production['ore_to_dore_ratio'] = df_production['ore_to_dore_ratio'].round(2)

# Load to MySQL
df_production.to_sql('production_summary', engine,
                     if_exists='append', index=False)

# Validate
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM production_summary")).scalar()
    print(f"production_summary loaded: {count} rows")
    result = conn.execute(text("""
        SELECT financial_year, ore_throughput_tonnes, recovery_rate_pct
        FROM production_summary
        ORDER BY financial_year
    """))
    print()
    print(f"{'Financial Year':<16} {'Ore Throughput':>18} {'Recovery Rate':>15}")
    print("-" * 52)
    for row in result:
        print(f"{row[0]:<16} {row[1]:>18,} {row[2]:>14.2f}%")

production_summary loaded: 3 rows

Financial Year       Ore Throughput   Recovery Rate
----------------------------------------------------
2021-22                   8,951,016          96.40%
2022-23                  14,642,594          98.14%
2023-24                  13,515,493          53.87%


In [7]:
# CELL 5
# Load both labour pillar CSVs into their respective MySQL tables.
# labour_quarterly contains the full 48-quarter time series of Queensland
# surface mineral mine headcounts from March 2014 to December 2025.
# labour_production_linked contains the annual productivity metrics
# linking workforce size to production volume across the three financial years.

import pandas as pd
import os

# ── labour_quarterly ────────────────────────────────────────────────────
quarterly_path = os.path.join('..', 'data', 'processed', 'labour',
                               'qld_surface_mineral_workers_quarterly.csv')
df_quarterly = pd.read_csv(quarterly_path)

# Ensure correct dtypes
df_quarterly['quarter_date'] = pd.to_datetime(df_quarterly['quarter_date']).dt.date
df_quarterly['total_surface_mineral_workers'] = df_quarterly[
    'total_surface_mineral_workers'].astype('Int64')

# Select only columns that match the table schema
df_quarterly_load = df_quarterly[[
    'quarter_date', 'quarter_label', 'financial_year',
    'total_surface_mineral_workers'
]]

df_quarterly_load.to_sql('labour_quarterly', engine,
                          if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM labour_quarterly")).scalar()
    print(f"labour_quarterly loaded: {count} rows")

# ── labour_production_linked ─────────────────────────────────────────────
linked_path = os.path.join('..', 'data', 'processed', 'labour',
                            'labour_production_linked.csv')
df_labour_linked = pd.read_csv(linked_path)

# Ensure correct dtypes
df_labour_linked['ore_throughput_tonnes'] = df_labour_linked[
    'ore_throughput_tonnes'].astype('Int64')
df_labour_linked['avg_workers'] = df_labour_linked['avg_workers'].astype('Int64')
df_labour_linked['max_workers'] = df_labour_linked['max_workers'].astype('Int64')
df_labour_linked['min_workers'] = df_labour_linked['min_workers'].astype('Int64')

# Select only columns that match the table schema
df_labour_linked_load = df_labour_linked[[
    'financial_year', 'ore_throughput_tonnes', 'dore_output_tonnes',
    'gold_mined_oz', 'gold_recovered_oz', 'ore_to_dore_ratio',
    'recovery_rate_pct', 'avg_workers', 'max_workers', 'min_workers',
    'tonnes_per_worker', 'workers_per_mtpa', 'gold_oz_per_worker'
]]

df_labour_linked_load.to_sql('labour_production_linked', engine,
                              if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM labour_production_linked")).scalar()
    print(f"labour_production_linked loaded: {count} rows")

    print()
    print("Labour productivity summary:")
    result = conn.execute(text("""
        SELECT financial_year, avg_workers,
               tonnes_per_worker, workers_per_mtpa,
               gold_oz_per_worker
        FROM labour_production_linked
        ORDER BY financial_year
    """))
    print(f"{'FY':<10} {'Avg Workers':>12} {'T/Worker':>10} "
          f"{'W/Mtpa':>10} {'Oz/Worker':>10}")
    print("-" * 55)
    for row in result:
        print(f"{row[0]:<10} {row[1]:>12,} {row[2]:>10,.0f} "
              f"{row[3]:>10.1f} {row[4]:>10.2f}")

labour_quarterly loaded: 48 rows
labour_production_linked loaded: 3 rows

Labour productivity summary:
FY          Avg Workers   T/Worker     W/Mtpa  Oz/Worker
-------------------------------------------------------
2021-22           6,800      1,316      759.7      74.48
2022-23           6,606      2,217      451.1      92.00
2023-24           6,252      2,162      462.6     106.40


In [9]:
# CELL 6
# Load all three safety CSVs into their respective MySQL tables.
# safety_monthly contains 162 months of incident frequency rate data
# from July 2012 to December 2025.
# safety_annual contains the aggregated annual safety performance metrics.
# safety_production_linked contains the three-pillar linkage table
# connecting safety performance to production volume and recovery rate.

import pandas as pd
import os
import numpy as np

# ── safety_monthly ───────────────────────────────────────────────────────
monthly_path = os.path.join('..', 'data', 'processed', 'safety',
                             'safety_monthly.csv')
df_safety_monthly = pd.read_csv(monthly_path)

# Ensure correct dtypes
df_safety_monthly['month'] = pd.to_datetime(df_safety_monthly['month']).dt.date
df_safety_monthly['serious_accidents'] = df_safety_monthly[
    'serious_accidents'].astype('Int64')
df_safety_monthly['hpi_count'] = df_safety_monthly['hpi_count'].astype('Int64')

# Select columns matching schema
df_safety_monthly_load = df_safety_monthly[[
    'month', 'financial_year', 'serious_accidents', 'hpi_count',
    'million_hours_worked', 'safr', 'safr_monthly_avg',
    'safr_3mth_rolling', 'safr_12mth_rolling',
    'hpifr', 'hpifr_monthly_avg', 'hpifr_3mth_rolling', 'hpifr_12mth_rolling'
]]

df_safety_monthly_load.to_sql('safety_monthly', engine,
                               if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM safety_monthly")).scalar()
    print(f"safety_monthly loaded: {count} rows")

# ── safety_annual ────────────────────────────────────────────────────────
annual_path = os.path.join('..', 'data', 'processed', 'safety',
                            'safety_annual.csv')
df_safety_annual = pd.read_csv(annual_path)

# Select columns matching schema
df_safety_annual_load = df_safety_annual[[
    'financial_year', 'total_serious_accidents', 'total_hpi_count',
    'total_million_hours', 'avg_safr', 'avg_hpifr',
    'peak_safr', 'peak_hpifr', 'months_reported',
    'safr_12mth_annual', 'hpifr_12mth_annual'
]]

df_safety_annual_load.to_sql('safety_annual', engine,
                              if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM safety_annual")).scalar()
    print(f"safety_annual loaded: {count} rows")

# ── safety_production_linked ─────────────────────────────────────────────
linked_path = os.path.join('..', 'data', 'processed', 'safety',
                            'safety_production_linked.csv')
df_safety_linked = pd.read_csv(linked_path)

# Replace inf values with None before loading
df_safety_linked = df_safety_linked.replace([np.inf, -np.inf], np.nan)

# Select columns matching schema
df_safety_linked_load = df_safety_linked[[
    'financial_year', 'total_serious_accidents', 'total_hpi_count',
    'avg_safr', 'avg_hpifr', 'ore_throughput_tonnes',
    'recovery_rate_pct', 'sa_per_million_tonnes',
    'hpi_per_million_tonnes', 'hpi_to_sa_ratio'
]]

# Ensure correct dtypes
df_safety_linked_load['ore_throughput_tonnes'] = df_safety_linked_load[
    'ore_throughput_tonnes'].astype('Int64')

df_safety_linked_load.to_sql('safety_production_linked', engine,
                              if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM safety_production_linked")).scalar()
    print(f"safety_production_linked loaded: {count} rows")

    print()
    print("Three-pillar linkage summary:")
    result = conn.execute(text("""
        SELECT financial_year, total_serious_accidents,
               total_hpi_count, avg_safr,
               ore_throughput_tonnes, recovery_rate_pct,
               sa_per_million_tonnes, hpi_to_sa_ratio
        FROM safety_production_linked
        WHERE ore_throughput_tonnes IS NOT NULL
        ORDER BY financial_year
    """))
    print(f"{'FY':<10} {'SAs':>5} {'HPIs':>6} {'SAFR':>8} "
          f"{'Ore (Mt)':>10} {'Recovery':>10} {'SA/Mt':>8} {'HPI:SA':>8}")
    print("-" * 70)
    for row in result:
        print(f"{row[0]:<10} {row[1]:>5} {row[2]:>6} {row[3]:>8.3f} "
              f"{row[4]/1e6:>10.2f} {row[5]:>9.2f}% "
              f"{row[6]:>8.3f} {row[7]:>8.1f}x")

safety_monthly loaded: 324 rows
safety_annual loaded: 28 rows
safety_production_linked loaded: 14 rows

Three-pillar linkage summary:
FY           SAs   HPIs     SAFR   Ore (Mt)   Recovery    SA/Mt   HPI:SA
----------------------------------------------------------------------
2021-22       14    181    1.188       8.95     96.40%    1.564     12.9x
2022-23       10    200    0.802      14.64     98.14%    0.683     20.0x
2023-24       18    229    1.459      13.52     53.87%    1.332     12.7x


In [10]:
# CELL 7
# Clean any duplicate rows that may have been introduced by running earlier
# notebooks multiple times, then run a full validation query across all tables.
# We use ROW_NUMBER() to identify and delete duplicates keeping only the first
# occurrence of each unique record.
# Final validation confirms row counts and referential integrity across all tables.

with engine.connect() as conn:

    # ── Remove duplicates from safety_monthly ───────────────────────────
    conn.execute(text("""
        DELETE FROM safety_monthly
        WHERE id NOT IN (
            SELECT min_id FROM (
                SELECT MIN(id) as min_id
                FROM safety_monthly
                GROUP BY month, financial_year
            ) AS subquery
        )
    """))
    conn.commit()

    # ── Remove duplicates from safety_annual ────────────────────────────
    conn.execute(text("""
        DELETE FROM safety_annual
        WHERE id NOT IN (
            SELECT min_id FROM (
                SELECT MIN(id) as min_id
                FROM safety_annual
                GROUP BY financial_year
            ) AS subquery
        )
    """))
    conn.commit()

    # ── Remove duplicates from safety_production_linked ─────────────────
    conn.execute(text("""
        DELETE FROM safety_production_linked
        WHERE id NOT IN (
            SELECT min_id FROM (
                SELECT MIN(id) as min_id
                FROM safety_production_linked
                GROUP BY financial_year
            ) AS subquery
        )
    """))
    conn.commit()

    print("Duplicates cleaned ✅")
    print()

    # ── Full database validation ─────────────────────────────────────────
    tables = [
        'ref_financial_years',
        'ref_skill_shortages',
        'production_summary',
        'labour_quarterly',
        'labour_production_linked',
        'safety_monthly',
        'safety_annual',
        'safety_production_linked'
    ]

    print(f"{'Table':<30} {'Rows':>8} {'Status':>10}")
    print("-" * 52)
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")).scalar()
        status = "✅ OK" if count > 0 else "⚠️ EMPTY"
        print(f"{table:<30} {count:>8,} {status:>10}")

    print()

    # ── Cross-pillar integrity check ─────────────────────────────────────
    print("Cross-pillar integrity check:")
    print()

    result = conn.execute(text("""
        SELECT
            fy.financial_year,
            fy.expansion_period,
            ps.ore_throughput_tonnes,
            ps.recovery_rate_pct,
            lp.avg_workers,
            lp.tonnes_per_worker,
            sa.total_serious_accidents,
            sa.avg_safr,
            sa.avg_hpifr
        FROM ref_financial_years fy
        LEFT JOIN production_summary ps
            ON fy.financial_year = ps.financial_year
        LEFT JOIN labour_production_linked lp
            ON fy.financial_year = lp.financial_year
        LEFT JOIN safety_annual sa
            ON fy.financial_year = sa.financial_year
        WHERE ps.financial_year IS NOT NULL
        ORDER BY fy.financial_year
    """))

    print(f"{'FY':<10} {'Exp':>5} {'Ore Mt':>8} {'Recov%':>8} "
          f"{'Workers':>9} {'T/Wkr':>8} {'SAs':>5} "
          f"{'SAFR':>8} {'HPIFR':>8}")
    print("-" * 75)
    for row in result:
        exp = "✅" if row[1] else "  "
        ore = f"{row[2]/1e6:.2f}" if row[2] else "N/A"
        print(f"{row[0]:<10} {exp:>5} {ore:>8} {row[3]:>7.2f}% "
              f"{row[4]:>9,} {row[5]:>8,.0f} {row[6]:>5} "
              f"{row[7]:>8.3f} {row[8]:>8.3f}")

Duplicates cleaned ✅

Table                              Rows     Status
----------------------------------------------------
ref_financial_years                  14       ✅ OK
ref_skill_shortages                  15       ✅ OK
production_summary                    3       ✅ OK
labour_quarterly                     48       ✅ OK
labour_production_linked              3       ✅ OK
safety_monthly                      162       ✅ OK
safety_annual                        14       ✅ OK
safety_production_linked             14       ✅ OK

Cross-pillar integrity check:

FY           Exp   Ore Mt   Recov%   Workers    T/Wkr   SAs     SAFR    HPIFR
---------------------------------------------------------------------------
2021-22        ✅     8.95   96.40%     6,800    1,316    14    1.188   16.268
2022-23        ✅    14.64   98.14%     6,606    2,217    10    0.802   16.900
2023-24        ✅    13.52   53.87%     6,252    2,162    18    1.459   18.510
